# Structured outputs with validate-and-retry, using agentcast

OpenAI's [Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs) feature is the right answer when you can use it: pass a JSON Schema (or Pydantic model), get back JSON that exactly matches it. No retries needed.

But there are real cases where Structured Outputs alone isn't enough:

1. **Validation rules JSON Schema can't express.** "The list of category weights must sum to 1.0". "`end_date` must be after `start_date`". "At least one of `email` or `phone` must be present". You need code, not a schema.
2. **Models or providers that don't support strict mode.** Older OpenAI models, fine-tunes, third-party endpoints behind the OpenAI-compatible API, local models. The output looks like JSON ninety percent of the time and you want a clean retry loop for the other ten.
3. **You want validation + retry as a reusable layer** that doesn't care which LLM you're calling.

[`agentcast`](https://pypi.org/project/agentcast-py/) is a small (~200 LOC, MIT, zero runtime deps) library that wraps any LLM call with: extract JSON → validate → on failure, append the error as feedback and retry. BYO-LLM, BYO-validator. This notebook shows how to drop it in alongside the OpenAI Python SDK.

## Setup

In [ ]:
%pip install -q openai agentcast-py pydantic python-dotenv

In [ ]:
import json

# agentcast: validate-and-retry wrapper for any LLM call
from agentcast import CastError, cast, extract_json
from agentcast import adapters as cast_adapters
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field, model_validator

load_dotenv()
client = OpenAI()
MODEL = "gpt-4o-mini"

## Example 1: a Pydantic model with a custom invariant

We want to extract a product description into a typed shape, plus enforce a rule that JSON Schema can't say on its own: the per-category sentiment scores must sum to 1.0.

In [ ]:
class CategorySentiment(BaseModel):
    label: str
    score: float = Field(ge=0.0, le=1.0)


class ProductReview(BaseModel):
    product_name: str
    overall_rating: int = Field(ge=1, le=5)
    summary: str
    sentiment_breakdown: list[CategorySentiment]

    @model_validator(mode="after")
    def _scores_sum_to_one(self):
        total = sum(c.score for c in self.sentiment_breakdown)
        if abs(total - 1.0) > 1e-3:
            raise ValueError(
                f"sentiment_breakdown scores must sum to 1.0 (got {total:.3f}). "
                f"Re-normalize the per-category scores so they total 1.0."
            )
        return self


REVIEW_TEXT = """
The new SoundPod 3 has fantastic battery life — I get a full week on a single
charge — and the noise cancellation is the best I've used at this price. The
ear cups feel a little plasticky and the bundled cable is too short, but those
are small gripes. Sound quality is rich and balanced. I'd give it 4 out of 5.
"""

## A plain OpenAI call: works on the happy path

This is the baseline. With `gpt-4o-mini` and a clear instruction, you'll usually get valid JSON back. But "usually" is doing a lot of work — the moment a model adds a markdown fence, an apology, or a sentiment breakdown that sums to 0.97, your downstream code breaks.

In [ ]:
def call_openai(messages):
    """Tiny adapter: messages in, string out. The shape agentcast expects."""
    completion = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.2,
    )
    return completion.choices[0].message.content

## Use agentcast to validate + retry

`cast()` calls your LLM, runs `extract_json` on the response (which handles fenced blocks and prose-wrapped JSON), then runs your validator. On failure it appends the error to the message history as feedback and retries up to `max_retries` times. The Pydantic adapter turns a `BaseModel` into the validator shape `cast()` wants.

In [ ]:
import asyncio


async def extract_review():
    review = await cast(
        llm=call_openai,
        validate=cast_adapters.pydantic(ProductReview),
        prompt=(
            "Extract a structured product review from the text below. "
            "Include a sentiment_breakdown list with categories like "
            "battery, sound_quality, build_quality, value — each with a score "
            "in [0, 1]. The scores MUST sum to exactly 1.0.\n\n"
            f"REVIEW:\n{REVIEW_TEXT}"
        ),
        max_retries=2,
    )
    return review


review = asyncio.run(extract_review())
print(json.dumps(review, indent=2))

## Example 2: see the retry loop fire

To see the retry behavior end-to-end, swap in a stub LLM that produces a deliberately broken first response and a valid second response. The validation error from attempt 1 is appended to the conversation as feedback before attempt 2.

In [ ]:
# Build a stub LLM that returns a different response each call.
class FakeLLM:
    def __init__(self, responses):
        self._responses = list(responses)
        self.calls = []

    def __call__(self, messages):
        self.calls.append(messages)
        return self._responses.pop(0)


# First response: scores sum to 0.85 (invalid). Second: corrected.
fake = FakeLLM(
    [
        json.dumps(
            {
                "product_name": "SoundPod 3",
                "overall_rating": 4,
                "summary": "Great battery, plasticky build.",
                "sentiment_breakdown": [
                    {"label": "battery", "score": 0.5},
                    {"label": "sound_quality", "score": 0.25},
                    {"label": "build_quality", "score": 0.10},
                ],
            }
        ),
        json.dumps(
            {
                "product_name": "SoundPod 3",
                "overall_rating": 4,
                "summary": "Great battery, plasticky build.",
                "sentiment_breakdown": [
                    {"label": "battery", "score": 0.45},
                    {"label": "sound_quality", "score": 0.30},
                    {"label": "build_quality", "score": 0.15},
                    {"label": "value", "score": 0.10},
                ],
            }
        ),
    ]
)


async def with_retry_visibility():
    return await cast(
        llm=fake,
        validate=cast_adapters.pydantic(ProductReview),
        prompt="Extract the structured review.",
        on_attempt=lambda info: print(
            f"  attempt {info['attempt']}: error = {info['error'][:80]}..."
            if info["error"]
            else f"  attempt {info['attempt']}: ok"
        ),
        max_retries=2,
    )


result = asyncio.run(with_retry_visibility())
print()
print("final:", json.dumps(result, indent=2))
print(f"\ntotal LLM calls made: {len(fake.calls)}")

## Example 3: the failure mode — exhausted retries

If validation never passes, `cast()` raises `CastError` with the full attempt history attached. You can log it, alert on it, or return a graceful fallback to your user.

In [ ]:
always_broken = FakeLLM(
    [
        '{"product_name": "X", "overall_rating": 99, "summary": "", "sentiment_breakdown": []}',
        '{"product_name": "X", "overall_rating": 99, "summary": "", "sentiment_breakdown": []}',
        '{"product_name": "X", "overall_rating": 99, "summary": "", "sentiment_breakdown": []}',
    ]
)

try:
    asyncio.run(
        cast(
            llm=always_broken,
            validate=cast_adapters.pydantic(ProductReview),
            prompt="Extract the structured review.",
            max_retries=2,
        )
    )
except CastError as e:
    print(f"raised CastError: {e}")
    print(f"\n{len(e.attempts)} attempt(s) recorded")
    for i, attempt in enumerate(e.attempts, 1):
        print(f"  attempt {i}: {attempt['error'][:100]}")

## Example 4: extract_json on its own

Sometimes you don't need the full retry loop — just "pull JSON out of this LLM string, ignore the prose around it." `extract_json` handles fenced ` ```json ... ``` ` blocks, language-less fences, top-level arrays, inline JSON in prose, and refusals (returns `None`).

In [ ]:
samples = [
    '{"answer": 42}',
    'Sure! Here is the answer:\n\n```json\n{"name": "Widget", "price": 29.99}\n```\n\nLet me know if you need anything else.',
    "I'm sorry, I cannot answer that.",
    'The result is { "valid": true, "score": 0.87 }. Please verify.',
    '[{"k": 1}, {"k": 2}]',
]

for s in samples:
    extracted = extract_json(s)
    print(f"{s[:50]!r:55}  ->  {extracted}")

## When to use this vs. native Structured Outputs

| Situation | Recommendation |
|---|---|
| Latest OpenAI model + plain JSON shape | **Native Structured Outputs** — strict mode is simpler and avoids the retry round-trip. |
| You need validation rules JSON Schema can't express | `agentcast` with a Pydantic `model_validator` (Example 1 above). |
| You're calling an older model, a fine-tune, or a non-OpenAI provider that doesn't support strict mode | `agentcast` — works against any `messages -> string` callable. |
| You want one validation layer that's portable across providers | `agentcast` — same code regardless of which LLM is behind `llm=`. |

The two compose well. Use native Structured Outputs for the JSON shape, then wrap the call in `cast()` with a validator that only checks your business invariants. The model gets schema enforcement for free, and your code gets clean retries when the invariant fails.

## Further reading

- [`agentcast` source + docs](https://github.com/MukundaKatta/AgentCastPy)
- [The agent reliability stack](https://mukundakatta.github.io/agent-stack/) — `agentcast` is one of five small libraries (fit, guard, snap, vet, cast) that fix the boring parts of long-running agents.
- [Companion dataset](https://huggingface.co/datasets/mukunda1729/llm-output-extraction-cases) — 20 messy LLM outputs with expected JSON, useful for testing your own extractor against.